# AAR on a real GPU (Colab T4)

Verifies the one thing this repository could not verify on a CPU-only
host: **does the GPU path actually work, and would the planner have
chosen the GPU?**

Enable **Runtime > Change runtime type > T4 GPU** first.

The last cell writes `aar_gpu_result.json`. Save it to Drive and copy it
back into the repository as `data/gpu/aar_gpu_result.json` so the finding
becomes a report entry rather than a claim. A GPU measurement that lives
only in a browser session is a rumour, not evidence.

In [ ]:
!pip install -q --upgrade pip

# AAR is NOT on PyPI yet, so there is nothing to `pip install aar-analytics`.
# Install from GitHub instead: the `name @ url` form is what lets pip
# resolve EXTRAS from a remote source, which the old `#egg=` syntax and a
# bare .zip download both fail to do.
#
# Pin to a tag or a commit SHA, not `main`, for any run whose numbers you
# intend to keep - `main` moves, and a result nobody can reproduce is a
# rumour. AAR_VERSION below records what was actually used.
AAR_VERSION = "main"   # e.g. "v0.1.0" or a full commit SHA

url = f"git+https://github.com/faizansk25/AAR.git@{AAR_VERSION}"
!pip install -q "aar-analytics[duckdb,polars,pandas] @ {url}"

import aar, importlib.metadata as md
print("AAR", aar.__version__, "from", md.version("aar-analytics"))
print("engines:", {m for m in ("pyarrow", "duckdb", "polars", "pandas")
                    if __import__("importlib.util", fromlist=["util"]).find_spec(m)})

In [ ]:
import subprocess, sys
print(subprocess.run(
    ["nvidia-smi"], capture_output=True, text=True).stdout or
    "nvidia-smi unavailable")

## Run the verification

Expect this to take a few minutes: it may install RAPIDS, and the
benchmark builds 2M rows per engine. Reduce with `AAR_GPU_ROWS=200000`
if you just want the smoke answer.

In [ ]:
!curl -sL -o gpu_verification.py \
  https://raw.githubusercontent.com/faizansk25/AAR/main/tools/gpu_verification.py
print(len(open("gpu_verification.py").read()), "bytes fetched")

In [ ]:
!python gpu_verification.py

## Read the result before trusting it

A T4 is compute capability 7.5: **no bfloat16**, and FP64 runs at 1/64 of
FP32. Work that wins on a T4 can lose on an A100. What this establishes is
that the path *works* and that the cost model picks sensibly - not that a
particular kernel is a good idea on every GPU.

In [ ]:
import json
r = json.load(open("aar_gpu_result.json"))
print("CUDA:", json.dumps(r["cuda"], indent=2))
print("\nGPU engines:")
for k, v in r["engines"].items():
    if "gpu" in k or v.get("available"):
        print(f"  {k:12s} {v}")
print("\nCross-engine agreement:")
for k, v in r["benchmark"].items():
    print(f"  {k:12s} {v}")
print("\nnotes:", r["errors"])

## Save the evidence

The notebook is ephemeral. Persist the result before the tab closes.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p "/content/drive/MyDrive/aar-gpu"
    !cp aar_gpu_result.json "/content/drive/MyDrive/aar-gpu/"
    print("saved to Drive/MyDrive/aar-gpu/aar_gpu_result.json")
except Exception as exc:
    print("Drive not available:", exc)
    print("Download aar_gpu_result.json manually before closing this tab")